# Fan-out join · **05 — Article images**

The paired bar chart in notebook 04 carries the argument, but it's the
wrong shape for two jobs:

1. **`paired_bars_fan_out_medium.png`** – the same chart redrawn for
   Medium's text column (about 700px wide, so I draw it at 2×, ~1,460px).
   Shorter labels, bigger type, the title split over two lines, so nothing
   shrinks below readable when Medium scales it down.
2. **`cover_landscape.png`** (1800×945) and **`social_square.png`**
   (1260×1260) – the cover and social cards, in the same design as my MoJ
   Crown Court series, so this piece sits with the rest of my Medium work.

Every number comes from notebook 03's outputs; none are typed in.

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import GOLD_DIR  # noqa: E402

IN = GOLD_DIR / "01_fan_out"
OUT = GOLD_DIR / "02_charts"
OUT.mkdir(parents=True, exist_ok=True)

cmp = pd.read_csv(IN / "fan_out_by_tf.csv")
contrast = pd.read_csv(IN / "contrast.csv").set_index("lookup")
correct_n, fanned_n = contrast.england_pathways.iloc[0], contrast.england_pathways.iloc[1]

PAPER, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#9a9891", "#e6e4de"
CORRECT, FANOUT = "#2a78d6", "#eb6834"
plt.rcParams.update({"font.family": "DejaVu Sans", "text.color": INK})


def short(name: str) -> str:
    if name == "Other - Other Services":
        return "Other services"
    return (name.removesuffix(" Service").removesuffix(" Services")
                .replace("Other - ", "Other ").replace(" and ", " & "))


def signed(x: float) -> str:
    return f"{x:+.2f}".replace("-", "−")


def tidy(ax) -> None:
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", length=0, colors=INK2)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)


def paired_bars(ax, d: pd.DataFrame, bar_h: float, gap_fs: float, label_fs: float) -> None:
    """Correct grain above, fan-out below, gap in points to the right of each pair."""
    d = d.reset_index(drop=True)
    for i, r in d.iterrows():
        ax.barh(i + bar_h / 2 + 0.02, r.pct_over_18_correct, height=bar_h, color=CORRECT, lw=0)
        ax.barh(i - bar_h / 2 - 0.02, r.pct_over_18_fanout, height=bar_h, color=FANOUT, lw=0)
        ax.text(max(r.pct_over_18_correct, r.pct_over_18_fanout) + 0.9, i, f"{signed(r.gap_points)} pts",
                va="center", ha="left", fontsize=gap_fs, color=INK2)
    ax.set_yticks(range(len(d)), [short(n) for n in d.tf_name], fontsize=label_fs, color=INK)
    ax.set_ylim(-0.65, len(d) - 0.35)
    tidy(ax)

## 1. The chart, at Medium width

In [2]:
d = cmp.sort_values("gap_points", ascending=True)      # largest positive gap at the top
fig, ax = plt.subplots(figsize=(7.3, 10.6), dpi=200)
fig.patch.set_facecolor(PAPER); ax.set_facecolor(PAPER)
paired_bars(ax, d, bar_h=0.36, gap_fs=9, label_fs=9.5)
ax.set_xlim(0, 56)
ax.set_xticks(range(0, 51, 10), [f"{x}%" for x in range(0, 51, 10)], fontsize=9)
ax.set_xlabel("Incomplete pathways waiting over 18 weeks, August 2026", fontsize=9.5, color=INK2)
ax.legend(handles=[Patch(color=CORRECT, label="At pathway grain (matches NHS England)"),
                   Patch(color=FANOUT, label="Through a one-to-many ICB lookup")],
          loc="lower left", bbox_to_anchor=(-0.02, 1.0), frameon=False, fontsize=9.5, handlelength=1.1)
fig.text(0.03, 0.985, "Same file, one extra join.\nNeither bar looks wrong.",
         fontsize=17, fontweight="bold", va="top", ha="left", linespacing=1.15)
fig.text(0.03, 0.915, f"% over 18 weeks by treatment function, England. The join repeats each trust's\n"
         f"rows once per ‘main’ ICB: {correct_n / 1e6:.2f}m pathways become {fanned_n / 1e6:.2f}m, "
         f"and no error is raised.\nGap = fan-out minus correct, percentage points.",
         fontsize=9.5, color=INK2, va="top", ha="left", linespacing=1.4)
fig.text(0.03, 0.008, "Source: NHS England, RTT Waiting Times, August 2026. Reported providers, NONC excluded.",
         fontsize=7.8, color=MUTED, va="bottom")
fig.tight_layout(rect=(0, 0.015, 1, 0.858))
fig.savefig(OUT / "paired_bars_fan_out_medium.png", facecolor=PAPER)
plt.show()

## 2. The cover and social cards

Same design as my MoJ Crown Court series cards: a 1800×945 landscape for
Medium's featured image and a 1260×1260 square for social posts. The
drawing lives in `src/make_cover_card.py`; the three stat-card numbers are
read from notebook 03's CSVs there.

In [3]:
from make_cover_card import build, figures  # noqa: E402

f = figures()
print(f)
for sq in (False, True):
    print("wrote", build(sq).name)

{'inflation': '×1.12', 'rate_move': '0.09 pts', 'rank': '6 of 23'}
wrote cover_landscape.png


wrote social_square.png


## Alt text

In [4]:
cards_alt = (
    "Cover card: 'The Join You Never Documented.' NHS data engineering, Medium, November 2026, by Yusuf Ismail. "
    f"Three figures: the denominator inflated {f['inflation']} with no error raised; the England rate moved "
    f"{f['rate_move']} and still looks right; {f['rank']} specialties change league-table rank. "
    "Pull quote: 'Same file. Same measure. No error. Two answers that both look right.'"
)
alt_path = OUT / "alt_text.md"
base = alt_path.read_text().split("\n## paired_bars_fan_out_medium.png")[0].rstrip()
medium_alt = base.split("## paired_bars_fan_out.png", 1)[1].strip()
alt_path.write_text(base + "\n\n## paired_bars_fan_out_medium.png\n\nSame description as paired_bars_fan_out.png:\n\n"
                    + medium_alt + "\n\n## cover_landscape.png and social_square.png\n\n" + cards_alt + "\n")
print(cards_alt)

Cover card: 'The Join You Never Documented.' NHS data engineering, Medium, November 2026, by Yusuf Ismail. Three figures: the denominator inflated ×1.12 with no error raised; the England rate moved 0.09 pts and still looks right; 6 of 23 specialties change league-table rank. Pull quote: 'Same file. Same measure. No error. Two answers that both look right.'
